# Qwen2.5-VL 7B FP16 Chart Summarization + NLLB Bangla Translation

This notebook is prepared for a cluster/Jupyter environment with internet access. It follows the original Colab-style flow: setup, device check, Model 1 (Qwen2.5-VL), Model 2 (NLLB), dataset loading, inference loop, and output saving.

The dataset has been split into 10 ZIP files. Run one part at a time for normal cluster jobs, or enable the all-parts loop if the session has enough time.

## Phase 0 - What to upload

Upload this whole folder to the cluster:

- `run_qwen25vl_7b_fp16_nllb_cluster.ipynb`
- `dataset_parts/part_01.zip` through `dataset_parts/part_10.zip`
- `manifests/` folder
- `merge_cluster_outputs.py`

Open this notebook from the uploaded folder so all relative paths work.

## Phase 1 - Install dependencies

In [ ]:
# Run this cell once per fresh cluster environment.
# If your cluster already has CUDA PyTorch installed, you can skip the torch install line.
%pip install -q --upgrade pip
%pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu126
%pip install -q "transformers>=4.49" "accelerate>=1.0" pandas openpyxl pillow tqdm


## Phase 2 - Imports and device check

In [ ]:
import gc
import os
import time
import zipfile
from pathlib import Path

import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoProcessor, AutoTokenizer, Qwen2_5_VLForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(torch.__version__)
    print(torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info(0)
    print(f"CUDA memory: {free / 1024**3:.2f} GiB free / {total / 1024**3:.2f} GiB total")


## Phase 3 - Run configuration

In [ ]:
# Normal use: keep RUN_ALL_PARTS = False and set PART_ID to 1..10.
# Long session use: set RUN_ALL_PARTS = True to process every part in order.
PART_ID = 1
RUN_ALL_PARTS = False

DATASET_ZIP_DIR = Path("dataset_parts")
WORK_DIR = Path("cluster_work")
OUTPUT_DIR = Path("cluster_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VL_MODEL_NAME = "Qwen/Qwen2.5-VL-7B-Instruct"
TRANS_MODEL_NAME = "facebook/nllb-200-distilled-600M"

# FP16 7B usually needs a larger cluster GPU than an 8 GB desktop card.
QWEN_MIN_PIXELS = 128 * 28 * 28
QWEN_MAX_PIXELS = 768 * 28 * 28
MAX_NEW_TOKENS_QWEN = 250
MAX_NEW_TOKENS_NLLB = 256

# Keep NLLB on CPU by default so the FP16 Qwen model owns GPU memory.
# Change to "cuda" only if the GPU has comfortable spare memory.
TRANSLATION_DEVICE = "cpu"

IMAGE_EXTS = {".png", ".jpg", ".jpeg"}
parts_to_run = list(range(1, 11)) if RUN_ALL_PARTS else [PART_ID]
parts_to_run


# Model - 1 : Chart Summary Qwen2.5-VL-7B-Instruct

In [ ]:
if device != "cuda":
    raise RuntimeError("CUDA is required for the FP16 Qwen2.5-VL 7B run.")

qwen_processor = AutoProcessor.from_pretrained(
    VL_MODEL_NAME,
    min_pixels=QWEN_MIN_PIXELS,
    max_pixels=QWEN_MAX_PIXELS,
)

if "qwen_model" not in globals() or qwen_model is None:
    qwen_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        VL_MODEL_NAME,
        torch_dtype=torch.float16,
        attn_implementation="sdpa",
        device_map={"": 0},
    )
    qwen_model.eval()

print("Qwen2.5-VL 7B FP16 loaded.")


# Model - 2: Translation Model NLLB200

`facebook/nllb-200-distilled-600M`

In [ ]:
if "trans_tokenizer" not in globals() or trans_tokenizer is None:
    trans_tokenizer = AutoTokenizer.from_pretrained(TRANS_MODEL_NAME)

if "trans_model" not in globals() or trans_model is None:
    trans_model = AutoModelForSeq2SeqLM.from_pretrained(TRANS_MODEL_NAME)
    trans_model.to(TRANSLATION_DEVICE)
    trans_model.eval()

print(f"NLLB translation model loaded on {TRANSLATION_DEVICE}.")


## Pipeline: Planner + Insight Extractor + Summarizer = Qwen2.5-VL-7B-Instruct

In [ ]:
SINGLE_PASS_PROMPT = 'You are an expert data analyst and data journalist.\n\nAnalyze the chart image carefully.\n\nThink silently through these steps (DO NOT output them):\n1. Identify chart type, domain, axes, units, and time range (if present)\n2. Extract key data insights (trends, comparisons, patterns, anomalies)\n3. Interpret domain meaning (causes, implications, real-world impact)\n\nThen produce ONLY the final answer:\n\nSummary:\nWrite a single, well-structured paragraph (4-6 sentences) that:\n- Starts with the main trend or takeaway\n- Includes at least one comparison or pattern\n- Mentions any anomaly or notable feature (if present)\n- Explains the real-world significance\n\nUse clear, confident, natural English. No bullet points. No extra text.'

def generate_chart_summary(image, max_new_tokens=MAX_NEW_TOKENS_QWEN):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": SINGLE_PASS_PROMPT},
            ],
        }
    ]
    text = qwen_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = qwen_processor(text=[text], images=[image], return_tensors="pt")
    inputs = {key: value.to("cuda") for key, value in inputs.items()}
    input_len = inputs["input_ids"].shape[1]

    with torch.inference_mode():
        output = qwen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
            temperature=None,
            top_p=None,
        )

    new_tokens = output[0][input_len:]
    return qwen_processor.decode(new_tokens, skip_special_tokens=True).strip()


def translate_en_to_bn(text, max_new_tokens=MAX_NEW_TOKENS_NLLB):
    inputs = trans_tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    inputs = {key: value.to(TRANSLATION_DEVICE) for key, value in inputs.items()}
    with torch.inference_mode():
        outputs = trans_model.generate(
            **inputs,
            forced_bos_token_id=trans_tokenizer.convert_tokens_to_ids("ben_Beng"),
            max_new_tokens=max_new_tokens,
        )
    return trans_tokenizer.decode(outputs[0], skip_special_tokens=True)


# Load Split Dataset

In [ ]:
def extract_part(part_id):
    zip_path = DATASET_ZIP_DIR / f"part_{part_id:02d}.zip"
    if not zip_path.exists():
        raise FileNotFoundError(f"Missing dataset ZIP: {zip_path}")
    extract_dir = WORK_DIR / f"part_{part_id:02d}"
    marker = extract_dir / ".extracted"
    if not marker.exists():
        extract_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path, "r") as archive:
            archive.extractall(extract_dir)
        marker.write_text("ok\n", encoding="utf-8")
    image_files = sorted(path for path in extract_dir.rglob("*") if path.suffix.lower() in IMAGE_EXTS)
    if not image_files:
        raise RuntimeError(f"No images found after extracting {zip_path}")
    return extract_dir, image_files


def metadata_from_name(path):
    parts = path.stem.split("_")
    split = parts[0] if parts else ""
    family = parts[1] if len(parts) >= 3 else ""
    numeric_id = int(parts[-1]) if parts and parts[-1].isdigit() else None
    image_id = f"{split}_{family}_{numeric_id}" if split and family and numeric_id is not None else path.stem
    return image_id, split, family, numeric_id


extract_dir, image_files = extract_part(parts_to_run[0])
df_images = pd.DataFrame({"image": [str(path) for path in image_files]})
df_images["image_name"] = df_images["image"].apply(lambda x: Path(x).name)
print(f"Preview part {parts_to_run[0]}: {len(df_images)} images")
df_images.head()


## New Dataset Creation by Single Prompt technique

In [ ]:
def process_part(part_id):
    extract_dir, image_files = extract_part(part_id)
    output_csv = OUTPUT_DIR / f"part_{part_id:02d}_qwen25vl7b_fp16_nllb.csv"
    output_xlsx = OUTPUT_DIR / f"part_{part_id:02d}_qwen25vl7b_fp16_nllb.xlsx"

    completed = set()
    rows = []
    if output_csv.exists():
        previous = pd.read_csv(output_csv)
        rows = previous.to_dict("records")
        completed = set(previous.loc[previous["status"] == "done", "image_id"].astype(str))
        print(f"Part {part_id:02d}: resuming with {len(completed)} completed rows.")

    for image_path in tqdm(image_files, desc=f"Part {part_id:02d}", unit="img"):
        image_id, split, family, numeric_id = metadata_from_name(image_path)
        if image_id in completed:
            continue

        started = time.perf_counter()
        row = {
            "part_id": part_id,
            "image_id": image_id,
            "image_name": image_path.name,
            "image_path": str(image_path),
            "split": split,
            "family": family,
            "numeric_id": numeric_id if numeric_id is not None else "",
            "english_summary": "",
            "bangla_summary": "",
            "status": "done",
            "error": "",
            "seconds": "",
        }

        try:
            with Image.open(image_path) as img:
                image = img.convert("RGB")
                english_summary = generate_chart_summary(image)
            bangla_summary = translate_en_to_bn(english_summary)
            row["english_summary"] = english_summary
            row["bangla_summary"] = bangla_summary
        except Exception as exc:
            row["status"] = "failed"
            row["error"] = f"{type(exc).__name__}: {exc}"
        finally:
            row["seconds"] = round(time.perf_counter() - started, 2)
            rows.append(row)
            pd.DataFrame(rows).drop_duplicates("image_id", keep="last").to_csv(output_csv, index=False, encoding="utf-8-sig")

            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()

    final_df = pd.DataFrame(rows).drop_duplicates("image_id", keep="last")
    final_df.to_csv(output_csv, index=False, encoding="utf-8-sig")
    final_df.to_excel(output_xlsx, index=False)
    print(f"Saved part {part_id:02d}: {output_csv}")
    print(f"Saved part {part_id:02d}: {output_xlsx}")
    return final_df


# Create Final Dataset & Save

In [ ]:
part_frames = []
for part_id in parts_to_run:
    part_frames.append(process_part(part_id))

final_dataset = pd.concat(part_frames, ignore_index=True) if part_frames else pd.DataFrame()
combined_csv = OUTPUT_DIR / "combined_current_session_qwen25vl7b_fp16_nllb.csv"
combined_xlsx = OUTPUT_DIR / "combined_current_session_qwen25vl7b_fp16_nllb.xlsx"
final_dataset.to_csv(combined_csv, index=False, encoding="utf-8-sig")
final_dataset.to_excel(combined_xlsx, index=False)
print("Final dataset created for this notebook session:")
print(combined_csv)
print(combined_xlsx)
final_dataset.head()


## Phase 7 - After all parts finish

Download or keep all `cluster_outputs/part_*.csv` files together. Then run:

```bash
python merge_cluster_outputs.py --input-dir cluster_outputs --output-prefix cluster_outputs/qwen25vl7b_fp16_nllb_full_dataset
```

That creates the full merged CSV and XLSX.